In [ ]:
from pathlib import Path
import os
import torch

cwd = Path.cwd().resolve()
root_candidates = [cwd, cwd / "inference" / "training" / "rtmdet"]
ROOT = next((path for path in root_candidates if (path / "config.py").is_file()), None)
assert ROOT is not None, "Could not locate the RTMDet working directory."
os.chdir(ROOT)

CONFIG_PATH = ROOT / "config.py"
CHECKPOINT_PATH = ROOT / "outputs" / "epoch_100.pth"
DATASET_ROOT = (ROOT / "../../dataset/22Circuit testing 2 -test.coco").resolve()
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

print("Config     :", CONFIG_PATH)
print("Checkpoint :", CHECKPOINT_PATH)
print("Dataset    :", DATASET_ROOT)
print("Device     :", DEVICE)

assert CONFIG_PATH.is_file(), f"Config file not found: {CONFIG_PATH}"
assert CHECKPOINT_PATH.is_file(), f"Checkpoint not found: {CHECKPOINT_PATH}"
assert (DATASET_ROOT / "test").is_dir(), f"Test dataset not found: {DATASET_ROOT / 'test'}"
print("Ready.")


In [ ]:
from mmdet.apis import init_detector

model = init_detector(
    str(CONFIG_PATH),
    str(CHECKPOINT_PATH),
    device=DEVICE,
)

print("RTMDet model loaded.")
print("Classes:", model.dataset_meta["classes"])


In [ ]:
from mmengine.config import Config
from mmengine.runner import Runner

# Load the portable training configuration and evaluate the saved checkpoint.
test_cfg = Config.fromfile(str(CONFIG_PATH))
test_cfg.load_from = str(CHECKPOINT_PATH)
test_cfg.work_dir = str(ROOT / "test_outputs")

runner = Runner.from_cfg(test_cfg)
coco_metrics = runner.test()

print("\n=== COCO Test Result ===")
print(coco_metrics)


In [ ]:
from mmdet.apis import inference_detector
import matplotlib.pyplot as plt
import cv2

test_images = sorted(
    list((DATASET_ROOT / "test").glob("*.jpg"))
    + list((DATASET_ROOT / "test").glob("*.jpeg"))
    + list((DATASET_ROOT / "test").glob("*.png"))
)
assert test_images, f"No test images found in: {DATASET_ROOT / 'test'}"
print("Test images:", len(test_images))

img_path = test_images[0]
result = inference_detector(model, str(img_path))

print("Image:", img_path.name)
print("Detected objects:", len(result.pred_instances))


In [ ]:
from mmdet.apis import inference_detector
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2

classes = model.dataset_meta["classes"]
CONF_THRESHOLD = 0.5
CLASS_COLORS = {
    "black_wire": "black",
    "capacitor": "magenta",
    "ic_chip": "green",
    "led": "gold",
    "red_wire": "red",
    "resistance330": "darkorange",
    "resistance470": "blue",
    "white_wire": "deepskyblue",
}

for img_path in test_images:
    result = inference_detector(model, str(img_path))
    image = cv2.imread(str(img_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    prediction = result.pred_instances.cpu()

    detections = [
        (bbox, score, label)
        for bbox, score, label in zip(
            prediction.bboxes.numpy(),
            prediction.scores.numpy(),
            prediction.labels.numpy(),
        )
        if score >= CONF_THRESHOLD
    ]

    fig, axis = plt.subplots(figsize=(17, 10))
    axis.imshow(image)
    fig.subplots_adjust(right=0.72)

    for index, (bbox, score, label) in enumerate(detections, start=1):
        x1, y1, x2, y2 = bbox
        class_name = classes[label]
        color = CLASS_COLORS[class_name]
        axis.add_patch(
            patches.Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                linewidth=2,
                edgecolor=color,
                facecolor="none",
            )
        )
        axis.text(
            x1,
            max(y1 - 4, 5),
            str(index),
            fontsize=9,
            fontweight="bold",
            color=color,
            bbox=dict(facecolor="white", edgecolor=color, alpha=0.85, pad=1),
        )

    fig.text(0.74, 0.92, "Detection Results", fontsize=14, fontweight="bold", verticalalignment="top")
    y_position = 0.87
    for index, (_, score, label) in enumerate(detections, start=1):
        class_name = classes[label]
        fig.text(
            0.74,
            y_position,
            f"{index:02d}. {class_name:<15} {score:.2f}",
            fontsize=11,
            family="monospace",
            color=CLASS_COLORS[class_name],
            fontweight="bold",
        )
        y_position -= 0.035

    axis.set_title(img_path.name)
    axis.axis("off")
    plt.show()


## Detection metrics and confusion matrix

Evaluation uses a confidence threshold of `0.5` and an IoU threshold of `0.5`. Predictions and ground-truth boxes are greedily matched once by IoU. Misclassifications are recorded by actual and predicted class, while unmatched predictions and ground truths are assigned to the `background` row and column.

- Precision = TP / (TP + FP)
- Recall = TP / (TP + FN)
- F1-score = 2 * Precision * Recall / (Precision + Recall)
- Detection accuracy = TP / (TP + FP + FN)

Object detection does not have a well-defined true-negative count, so detection accuracy is reported instead of standard classification accuracy. Use the earlier `Runner.test()` output for COCO mAP.


In [ ]:
import json
import time
from collections import defaultdict

import numpy as np
from mmdet.apis import inference_detector

CONF_THRESHOLD = 0.5
IOU_THRESHOLD = 0.5
ANNOTATION_PATH = DATASET_ROOT / "test" / "_annotations.coco.json"
METRICS_DIR = ROOT / "metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

with open(ANNOTATION_PATH, "r", encoding="utf-8") as file:
    coco = json.load(file)

classes = tuple(model.dataset_meta["classes"])
num_classes = len(classes)
background_index = num_classes
matrix_labels = list(classes) + ["background"]

category_name_by_id = {
    category["id"]: category["name"]
    for category in coco["categories"]
    if category["name"] in classes
}
class_index_by_name = {name: index for index, name in enumerate(classes)}

annotations_by_image = defaultdict(list)
for annotation in coco["annotations"]:
    category_name = category_name_by_id.get(annotation["category_id"])
    if category_name is not None and not annotation.get("iscrowd", 0):
        annotations_by_image[annotation["image_id"]].append(annotation)


def xywh_to_xyxy(box):
    x, y, width, height = box
    return [x, y, x + width, y + height]


def box_iou_matrix(boxes1, boxes2):
    boxes1 = np.asarray(boxes1, dtype=np.float32).reshape(-1, 4)
    boxes2 = np.asarray(boxes2, dtype=np.float32).reshape(-1, 4)
    if len(boxes1) == 0 or len(boxes2) == 0:
        return np.zeros((len(boxes1), len(boxes2)), dtype=np.float32)

    top_left = np.maximum(boxes1[:, None, :2], boxes2[None, :, :2])
    bottom_right = np.minimum(boxes1[:, None, 2:], boxes2[None, :, 2:])
    intersection_wh = np.clip(bottom_right - top_left, 0, None)
    intersection = intersection_wh[..., 0] * intersection_wh[..., 1]

    area1 = np.clip(boxes1[:, 2] - boxes1[:, 0], 0, None) * np.clip(
        boxes1[:, 3] - boxes1[:, 1], 0, None
    )
    area2 = np.clip(boxes2[:, 2] - boxes2[:, 0], 0, None) * np.clip(
        boxes2[:, 3] - boxes2[:, 1], 0, None
    )
    union = area1[:, None] + area2[None, :] - intersection
    return intersection / np.clip(union, 1e-9, None)


confusion_matrix = np.zeros((num_classes + 1, num_classes + 1), dtype=np.int64)
latencies_ms = []

# Warm up the model before measuring latency.
warmup_image_path = DATASET_ROOT / "test" / coco["images"][0]["file_name"]
for _ in range(3):
    _ = inference_detector(model, str(warmup_image_path))
if torch.cuda.is_available():
    torch.cuda.synchronize()

for image_number, image_info in enumerate(coco["images"], start=1):
    image_path = DATASET_ROOT / "test" / image_info["file_name"]
    if not image_path.is_file():
        raise FileNotFoundError(f"Test image not found: {image_path}")

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start_time = time.perf_counter()
    result = inference_detector(model, str(image_path))
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    latencies_ms.append((time.perf_counter() - start_time) * 1000)

    prediction = result.pred_instances.cpu()
    scores = prediction.scores.numpy()
    keep = scores >= CONF_THRESHOLD
    pred_boxes = prediction.bboxes.numpy()[keep]
    pred_labels = prediction.labels.numpy()[keep].astype(int)
    pred_scores = scores[keep]

    order = np.argsort(-pred_scores)
    pred_boxes = pred_boxes[order]
    pred_labels = pred_labels[order]

    gt_annotations = annotations_by_image[image_info["id"]]
    gt_boxes = np.asarray(
        [xywh_to_xyxy(annotation["bbox"]) for annotation in gt_annotations],
        dtype=np.float32,
    ).reshape(-1, 4)
    gt_labels = np.asarray(
        [
            class_index_by_name[category_name_by_id[annotation["category_id"]]]
            for annotation in gt_annotations
        ],
        dtype=int,
    )

    ious = box_iou_matrix(pred_boxes, gt_boxes)
    matched_gt = set()

    for pred_index, pred_label in enumerate(pred_labels):
        available_gt = [index for index in range(len(gt_boxes)) if index not in matched_gt]
        if available_gt:
            available_ious = ious[pred_index, available_gt]
            best_position = int(np.argmax(available_ious))
            best_gt_index = available_gt[best_position]
            best_iou = float(available_ious[best_position])
        else:
            best_gt_index = -1
            best_iou = 0.0

        if best_iou >= IOU_THRESHOLD:
            confusion_matrix[gt_labels[best_gt_index], pred_label] += 1
            matched_gt.add(best_gt_index)
        else:
            confusion_matrix[background_index, pred_label] += 1

    for gt_index, gt_label in enumerate(gt_labels):
        if gt_index not in matched_gt:
            confusion_matrix[gt_label, background_index] += 1

    print(f"[{image_number:02d}/{len(coco['images'])}] {image_info['file_name']}")

true_positive = np.diag(confusion_matrix[:num_classes, :num_classes]).astype(float)
false_positive = confusion_matrix[:, :num_classes].sum(axis=0) - true_positive
false_negative = confusion_matrix[:num_classes, :].sum(axis=1) - true_positive
support = confusion_matrix[:num_classes, :].sum(axis=1)


def safe_divide(numerator, denominator):
    return np.divide(
        numerator,
        denominator,
        out=np.zeros_like(np.asarray(numerator, dtype=float)),
        where=np.asarray(denominator) != 0,
    )


precision = safe_divide(true_positive, true_positive + false_positive)
recall = safe_divide(true_positive, true_positive + false_negative)
f1_score = safe_divide(2 * precision * recall, precision + recall)
detection_accuracy = safe_divide(
    true_positive, true_positive + false_positive + false_negative
)

micro_tp = true_positive.sum()
micro_fp = false_positive.sum()
micro_fn = false_negative.sum()
micro_precision = float(safe_divide(micro_tp, micro_tp + micro_fp))
micro_recall = float(safe_divide(micro_tp, micro_tp + micro_fn))
micro_f1 = float(safe_divide(2 * micro_precision * micro_recall, micro_precision + micro_recall))
micro_accuracy = float(safe_divide(micro_tp, micro_tp + micro_fp + micro_fn))

rows = []
for index, class_name in enumerate(classes):
    rows.append({
        "class": class_name,
        "support": int(support[index]),
        "tp": int(true_positive[index]),
        "fp": int(false_positive[index]),
        "fn": int(false_negative[index]),
        "accuracy": float(detection_accuracy[index]),
        "precision": float(precision[index]),
        "recall": float(recall[index]),
        "f1_score": float(f1_score[index]),
    })

rows.extend([
    {
        "class": "macro_average",
        "support": int(support.sum()),
        "tp": int(micro_tp),
        "fp": int(micro_fp),
        "fn": int(micro_fn),
        "accuracy": float(detection_accuracy.mean()),
        "precision": float(precision.mean()),
        "recall": float(recall.mean()),
        "f1_score": float(f1_score.mean()),
    },
    {
        "class": "micro_average",
        "support": int(support.sum()),
        "tp": int(micro_tp),
        "fp": int(micro_fp),
        "fn": int(micro_fn),
        "accuracy": micro_accuracy,
        "precision": micro_precision,
        "recall": micro_recall,
        "f1_score": micro_f1,
    },
])

header = (
    f"{'class':<18} {'support':>7} {'TP':>5} {'FP':>5} {'FN':>5} "
    f"{'accuracy':>9} {'precision':>9} {'recall':>9} {'F1':>9}"
)
print("\n" + header)
print("-" * len(header))
for row in rows:
    print(
        f"{row['class']:<18} {row['support']:>7d} {row['tp']:>5d} "
        f"{row['fp']:>5d} {row['fn']:>5d} {row['accuracy']:>9.4f} "
        f"{row['precision']:>9.4f} {row['recall']:>9.4f} {row['f1_score']:>9.4f}"
    )

matched_count = int(confusion_matrix[:num_classes, :num_classes].sum())
matched_class_accuracy = (
    float(true_positive.sum() / matched_count) if matched_count else 0.0
)

latencies_ms = np.asarray(latencies_ms, dtype=float)
processing_time = {
    "device": "cuda:0" if torch.cuda.is_available() else "cpu",
    "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
    "batch_size": 1,
    "warmup_runs": 3,
    "total_seconds": float(latencies_ms.sum() / 1000),
    "mean_ms": float(latencies_ms.mean()),
    "median_ms": float(np.median(latencies_ms)),
    "p95_ms": float(np.percentile(latencies_ms, 95)),
    "min_ms": float(latencies_ms.min()),
    "max_ms": float(latencies_ms.max()),
    "fps": float(1000 / latencies_ms.mean()),
}

print("\n=== Processing time (batch size 1) ===")
print(f"Device : {processing_time['device_name']}")
print(f"Total  : {processing_time['total_seconds']:.3f} s / {len(latencies_ms)} images")
print(f"Mean   : {processing_time['mean_ms']:.2f} ms/image")
print(f"Median : {processing_time['median_ms']:.2f} ms/image")
print(f"P95    : {processing_time['p95_ms']:.2f} ms/image")
print(f"Min/Max: {processing_time['min_ms']:.2f} / {processing_time['max_ms']:.2f} ms")
print(f"FPS    : {processing_time['fps']:.2f}")

metrics_summary = {
    "confidence_threshold": CONF_THRESHOLD,
    "iou_threshold": IOU_THRESHOLD,
    "num_images": len(coco["images"]),
    "num_ground_truth_objects": int(support.sum()),
    "processing_time": processing_time,
    "micro_average": rows[-1],
    "macro_average": rows[-2],
    "matched_class_accuracy": matched_class_accuracy,
    "per_class": rows[:-2],
    "confusion_matrix_labels": matrix_labels,
    "confusion_matrix": confusion_matrix.tolist(),
}

metrics_path = METRICS_DIR / "detection_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as file:
    json.dump(metrics_summary, file, ensure_ascii=False, indent=2)

print(f"\nMetrics saved to: {metrics_path}")
print(f"Matched-object class accuracy: {matched_class_accuracy:.4f}")

## Processing time

Run this cell after loading the model to measure end-to-end latency and FPS over all 22 test images. Timing uses batch size 1, three warm-up runs, and CUDA synchronization.


In [ ]:
import json
import time

import numpy as np
import torch
from mmdet.apis import inference_detector

TIMING_WARMUP_RUNS = 3
timing_images = sorted(
    list((DATASET_ROOT / "test").glob("*.jpg"))
    + list((DATASET_ROOT / "test").glob("*.jpeg"))
    + list((DATASET_ROOT / "test").glob("*.png"))
)
assert timing_images, f"No test images found in: {DATASET_ROOT / 'test'}"

print(f"Warm-up: {TIMING_WARMUP_RUNS} runs")
for _ in range(TIMING_WARMUP_RUNS):
    _ = inference_detector(model, str(timing_images[0]))
if torch.cuda.is_available():
    torch.cuda.synchronize()

processing_latencies_ms = []
for index, image_path in enumerate(timing_images, start=1):
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    started_at = time.perf_counter()
    _ = inference_detector(model, str(image_path))
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    processing_latencies_ms.append((time.perf_counter() - started_at) * 1000)
    print(f"[{index:02d}/{len(timing_images)}] {processing_latencies_ms[-1]:.2f} ms  {image_path.name}")

processing_latencies_ms = np.asarray(processing_latencies_ms, dtype=float)
processing_time = {
    "device": "cuda:0" if torch.cuda.is_available() else "cpu",
    "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
    "batch_size": 1,
    "warmup_runs": TIMING_WARMUP_RUNS,
    "num_images": len(timing_images),
    "total_seconds": float(processing_latencies_ms.sum() / 1000),
    "mean_ms": float(processing_latencies_ms.mean()),
    "median_ms": float(np.median(processing_latencies_ms)),
    "p95_ms": float(np.percentile(processing_latencies_ms, 95)),
    "min_ms": float(processing_latencies_ms.min()),
    "max_ms": float(processing_latencies_ms.max()),
    "fps": float(1000 / processing_latencies_ms.mean()),
}

print("\n=== Processing time (batch size 1) ===")
print(f"Device : {processing_time['device_name']}")
print(f"Total  : {processing_time['total_seconds']:.3f} s / {processing_time['num_images']} images")
print(f"Mean   : {processing_time['mean_ms']:.2f} ms/image")
print(f"Median : {processing_time['median_ms']:.2f} ms/image")
print(f"P95    : {processing_time['p95_ms']:.2f} ms/image")
print(f"Min/Max: {processing_time['min_ms']:.2f} / {processing_time['max_ms']:.2f} ms")
print(f"FPS    : {processing_time['fps']:.2f}")

metrics_dir = ROOT / "metrics"
metrics_dir.mkdir(parents=True, exist_ok=True)
metrics_path = metrics_dir / "detection_metrics.json"
if metrics_path.is_file():
    with metrics_path.open("r", encoding="utf-8") as file:
        saved_metrics = json.load(file)
else:
    saved_metrics = {}
saved_metrics["processing_time"] = processing_time
with metrics_path.open("w", encoding="utf-8") as file:
    json.dump(saved_metrics, file, ensure_ascii=False, indent=2)
print(f"Saved to: {metrics_path}")


In [ ]:
import matplotlib.pyplot as plt

row_totals = confusion_matrix.sum(axis=1, keepdims=True)
normalized_confusion = np.divide(
    confusion_matrix,
    row_totals,
    out=np.zeros_like(confusion_matrix, dtype=float),
    where=row_totals != 0,
)

fig, axes = plt.subplots(1, 2, figsize=(22, 9))
plots = [
    (confusion_matrix, "Detection confusion matrix (counts)", "d"),
    (normalized_confusion, "Detection confusion matrix (row normalized)", ".2f"),
]

for axis, (matrix, title, number_format) in zip(axes, plots):
    image = axis.imshow(matrix, cmap="Blues", vmin=0)
    axis.set_title(title, fontsize=14)
    axis.set_xlabel("Predicted class")
    axis.set_ylabel("Actual class")
    axis.set_xticks(range(len(matrix_labels)), matrix_labels, rotation=45, ha="right")
    axis.set_yticks(range(len(matrix_labels)), matrix_labels)

    threshold = matrix.max() / 2 if matrix.size else 0
    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            value = matrix[row, column]
            text = f"{int(value)}" if number_format == "d" else f"{value:.2f}"
            axis.text(
                column,
                row,
                text,
                ha="center",
                va="center",
                fontsize=8,
                color="white" if value > threshold else "black",
            )
    fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)

fig.tight_layout()
confusion_path = (Path(METRICS_DIR) / "confusion_matrix.png").resolve()
confusion_path.parent.mkdir(parents=True, exist_ok=True)
print(f"Saving confusion matrix to: {confusion_path}")

# Use a file object for reliable PNG output on Windows.
# Use a file object for reliable PNG output on Windows.
with confusion_path.open("wb") as image_file:
    fig.savefig(image_file, format="png", dpi=180, bbox_inches="tight")

plt.show()
print(f"Confusion matrix saved to: {confusion_path}")